In [1]:
# analise de vendas por região 

import pandas as pd
import numpy as np

df_vendas = pd.read_csv("dataset_pratica_limpeza_200k.csv")
display(df_vendas)
display(df_vendas.columns)
display(df_vendas.describe())
df_analise = df_vendas.copy()

# ================================
# 1. REMOÇÃO DE DUPLICADOS
# ================================
df_analise = df_analise.drop_duplicates(subset=["id_venda"])


# ================================
# 2. TRATAMENTO DE VALORES INVÁLIDOS
# ================================
df_analise.loc[df_analise["quantidade"] <= 0, "quantidade"] = None
df_analise.loc[df_analise["preco_unitario"] <= 0, "preco_unitario"] = None

df_analise = df_analise.dropna(
    subset=["quantidade", "preco_unitario", "id_venda"]
)


# ================================
# 3. TRATAMENTO DE VARIÁVEIS CATEGÓRICAS
# ================================
df_analise["status_pedido"] = df_analise["status_pedido"].fillna("Presencial")
df_analise["categoria"] = df_analise["categoria"].fillna("Não informado")
df_analise["produto"] = df_analise["produto"].fillna("Não informado")
df_analise["nome_cliente"] = df_analise["nome_cliente"].fillna("Não informado")
df_analise["cidade"] = df_analise["cidade"].fillna("Não informado")
df_analise["estado"] = df_analise["estado"].fillna("Não informado")


df_analise["categoria"] = df_analise["categoria"].str.strip().str.title()
df_analise["produto"] = df_analise["produto"].str.strip().str.title()
df_analise["estado"] = df_analise["estado"].str.strip().str.upper()

# ================================
# 4. TRATAMENTO DE IDADE
# ================================
df_analise["idade"] = df_analise["idade"].fillna(-1)

df_analise.loc[
    (df_analise["idade"] > 100) | (df_analise["idade"] <= 0),
    "idade"
] = -1


# ================================
# 5. TRATAMENTO DE DATAS
# ================================
df_analise["data_compra"] = pd.to_datetime(
    df_analise["data_compra"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

df_analise["data_valida"] = df_analise["data_compra"].notna()


# ================================
# 6. FEATURE ENGINEERING
# ================================
df_analise["valor_total"] = (
    df_analise["quantidade"] * df_analise["preco_unitario"]
)
df_analise["ticket_medio"] = (
    df_analise["valor_total"] / df_analise["quantidade"]
)
dic_regiao = {
'RJ': "Sudeste",
'SP': "Sudeste",
'PR': "Sul", 
'Não informado': None
,'RS': "Sul",
'MG': "Sudeste"
}
df_analise["regiao"] = df_analise["estado"].map(dic_regiao)

# ================================
# 7. VALIDAÇÃO FINAL
# ================================
display(df_analise["data_compra"].isna().sum())
display(df_analise)

# 1. ORDENAÇÃO DOS DADOS

# Ordenando por maior valor de venda e, em caso de empate, pela data mais recente
df_analise = df_analise.sort_values(
    by=["valor_total", "data_compra"],
    ascending=[False, False]
)

# Resetando o índice após ordenação
df_analise = df_analise.reset_index(drop=True)

display(df_analise)



# 2. FILTRO POR REGIÃO

# Criando DataFrames filtrados por região

df_vendas_sudeste = df_analise[
    df_analise["regiao"] == "Sudeste"
]

df_vendas_sul = df_analise[
    df_analise["regiao"] == "Sul"
]

# Registros onde a região não foi identificada (NaN)
df_vendas_desconhecido = df_analise[
    df_analise["regiao"].isna()
]


# 3. VISUALIZAÇÃO DOS RESULTADOS

display(df_vendas_sudeste)
display(df_vendas_sul)
display(df_vendas_desconhecido)

# 1. FILTRO POR PRODUTO E REGIÃO
# Vendas do Produto A na região Sudeste
df_vendas_A_sudeste = df_analise[
    (df_analise["produto"] == "Produto A") &
    (df_analise["regiao"] == "Sudeste")
]


# 2. FILTRO POR PERÍODO (ANO)
# Vendas realizadas no ano de 2024
df_vendas_2024 = df_analise[
    df_analise["data_compra"].dt.year == 2024
]

# 3. VISUALIZAÇÃO DOS RESULTADO
display(df_vendas_A_sudeste)
display(df_vendas_2024)
# ================================
# 1. RANKING DE FATURAMENTO POR ESTADO
# ================================
df_analise_estado = (
    df_analise[["estado", "valor_total"]]
    .groupby("estado")
    .sum()
    .sort_values(by="valor_total", ascending=False)
    .reset_index()
)

df_analise_estado["valor_total"] = df_analise_estado["valor_total"].map(
    lambda x: f"R$ {x:,.2f}"
)


# ================================
# 2. PRODUTOS MAIS VENDIDOS NO PR
# ================================
df_analise_PR_produtos = (
    df_analise[df_analise["estado"] == "PR"]
    .groupby("produto", as_index=False)["quantidade"]
    .sum()
    .sort_values(by="quantidade", ascending=False)
)


# ================================
# 3. ANÁLISE CRUZADA (ESTADO x PRODUTO)
# ================================
df_produto_estado = (
    df_analise[["estado", "produto", "quantidade"]]
    .groupby(["estado", "produto"])
    .sum()
)

df_estado_produto = (
    df_analise[["estado", "produto", "quantidade"]]
    .groupby(["produto", "estado"])
    .sum()
)


# ================================
# 4. VISUALIZAÇÃO
# ================================
display(df_produto_estado)
display(df_estado_produto)
display(df_analise_estado)
display(df_analise_PR_produtos)
df_pivot = df_analise.pivot_table(
    values="quantidade",
    index="estado",
    columns="produto",
    aggfunc="sum",
    fill_value=0
)

display(df_pivot)


,id_venda,cliente_id,nome_cliente,idade,cidade,estado,categoria,produto,preco_unitario,quantidade,data_compra,status_pedido
0,1,2824.0,NaN,NaN,Rio de Janeiro,RJ,Alimentos,Produto A,594.59,1.0,2022-06-14 08:28:15,NaN
1,2,NaN,Bruno,-1.0,São Paulo,RJ,Móveis,Produto C,285.09,11.0,2021-01-14 13:49:57,Cancelado
2,3,NaN,NaN,200.0,Porto Alegre,SP,Móveis,Produto A,-50.00,NaN,2022-02-26 02:31:56,NaN
3,4,NaN,Ana,NaN,São Paulo,PR,Eletrônicos,NaN,NaN,NaN,NaN,NaN
4,5,2169.0,Eduardo,NaN,Rio de Janeiro,PR,Móveis,Produto C,-50.00,18.0,2023-08-21 02:02:11,Concluído
...,...,...,...,...,...,...,...,...,...,...,...,...
199995,199996,1355.0,Eduardo,-1.0,São Paulo,NaN,Móveis,Produto A,NaN,10.0,NaN,Em transporte
199996,199997,9348.0,Eduardo,200.0,Porto Alegre,SP,Móveis,Produto B,NaN,-3.0,2020-03-01 15:54:55,Concluído
199997,199998,4396.0,Carlos,NaN,Belo Horizonte,PR,Alimentos,Produto C,-50.00,-3.0,NaN,NaN
199998,199999,NaN,NaN,-1.0,Belo Horizonte,NaN,Alimentos,Produto A,-50.00,NaN,NaN,NaN


Index(['id_venda', 'cliente_id', 'nome_cliente', 'idade', 'cidade', 'estado',
       'categoria', 'produto', 'preco_unitario', 'quantidade', 'data_compra',
       'status_pedido'],
      dtype='str')

,id_venda,cliente_id,idade,preco_unitario,quantidade
count,200000.000000,100079.000000,150180.000000,133495.000000,133508.000000
mean,100000.500000,5504.532799,82.651165,226.983385,3.792612
std,57735.171256,2599.235711,85.998828,343.318514,7.895942
min,1.000000,1000.000000,-1.000000,-50.000000,-3.000000
25%,50000.750000,3254.000000,-1.000000,-50.000000,-3.000000
50%,100000.500000,5508.000000,49.000000,-50.000000,1.000000
75%,150000.250000,7750.000000,200.000000,504.095000,11.000000
max,200000.000000,9999.000000,200.000000,999.990000,20.000000


np.int64(11317)

,id_venda,cliente_id,nome_cliente,idade,cidade,estado,categoria,produto,preco_unitario,quantidade,data_compra,status_pedido,data_valida,valor_total,ticket_medio,regiao
0,1,2824.0,Não informado,-1.0,Rio de Janeiro,RJ,Alimentos,Produto A,594.59,1.0,2022-06-14 08:28:15,Presencial,True,594.59,594.59,Sudeste
1,2,NaN,Bruno,-1.0,São Paulo,RJ,Móveis,Produto C,285.09,11.0,2021-01-14 13:49:57,Cancelado,True,3135.99,285.09,Sudeste
17,18,8811.0,Ana,79.0,Rio de Janeiro,SP,Beleza,Produto A,678.46,13.0,2022-10-05 21:04:40,Presencial,True,8819.98,678.46,Sudeste
21,22,NaN,Não informado,21.0,Não informado,PR,Eletrônicos,Produto A,13.51,9.0,NaT,Presencial,False,121.59,13.51,Sul
23,24,5088.0,Carlos,-1.0,Porto Alegre,NÃO INFORMADO,Roupas,Produto B,865.96,14.0,NaT,Presencial,False,12123.44,865.96,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199973,199974,1136.0,Não informado,79.0,São Paulo,PR,Roupas,Não Informado,998.16,15.0,2022-01-24 00:45:54,Em transporte,True,14972.40,998.16,Sul
199976,199977,NaN,Eduardo,-1.0,Belo Horizonte,RJ,Eletrônicos,Produto B,254.99,5.0,2024-09-06 23:41:53,Cancelado,True,1274.95,254.99,Sudeste
199987,199988,NaN,Daniela,-1.0,Não informado,SP,Roupas,Não Informado,836.64,2.0,NaT,Cancelado,False,1673.28,836.64,Sudeste
199991,199992,NaN,Ana,-1.0,Porto Alegre,MG,Roupas,Produto C,455.78,14.0,NaT,Em transporte,False,6380.92,455.78,Sudeste


...

quantidade
estado        produto                  
MG            Não Informado      9250.0
              Produto A         10077.0
              Produto B          9547.0
              Produto C          9924.0
NÃO INFORMADO Não Informado      9781.0
              Produto A          9708.0
              Produto B         10157.0
              Produto C          9846.0
PR            Não Informado      9544.0
              Produto A          9440.0
              Produto B         10154.0
              Produto C         10127.0
RJ            Não Informado      9711.0
              Produto A          9661.0
              Produto B         10043.0
              Produto C         10152.0
RS            Não Informado      8953.0
              Produto A          9843.0
              Produto B         10392.0
              Produto C         10185.0
SP            Não Informado     10131.0
              Produto A          9499.0
              Produto B          9873.0
              Produto C          9328.0

quantidade
produto       estado                   
Não Informado MG                 9250.0
              NÃO INFORMADO      9781.0
              PR                 9544.0
              RJ                 9711.0
              RS                 8953.0
              SP                10131.0
Produto A     MG                10077.0
              NÃO INFORMADO      9708.0
              PR                 9440.0
              RJ                 9661.0
              RS                 9843.0
              SP                 9499.0
Produto B     MG                 9547.0
              NÃO INFORMADO     10157.0
              PR                10154.0
              RJ                10043.0
              RS                10392.0
              SP                 9873.0
Produto C     MG                 9924.0
              NÃO INFORMADO      9846.0
              PR                10127.0
              RJ                10152.0
              RS                10185.0
              SP                 9328.0

,estado,valor_total
0,PR,"R$ 20,267,664.03"
1,RS,"R$ 19,974,646.98"
2,RJ,"R$ 19,901,911.32"
3,NÃO INFORMADO,"R$ 19,712,358.54"
4,MG,"R$ 19,604,670.81"
5,SP,"R$ 19,544,132.35"


,produto,quantidade
2,Produto B,10154.0
3,Produto C,10127.0
0,Não Informado,9544.0
1,Produto A,9440.0


produto,Não Informado,Produto A,Produto B,Produto C
estado,,,,
MG,9250.0,10077.0,9547.0,9924.0
NÃO INFORMADO,9781.0,9708.0,10157.0,9846.0
PR,9544.0,9440.0,10154.0,10127.0
RJ,9711.0,9661.0,10043.0,10152.0
RS,8953.0,9843.0,10392.0,10185.0
SP,10131.0,9499.0,9873.0,9328.0
